In [5]:
import os

# ── Configuration ───────────────────────────────────────────────────────────
# Accuracies are read straight from the per-run *_metrics.json artifacts
# (same discovery that builds the 'acc' column of the d_exact tables written
# by scripts/slurm/run_score_subsets.sh), so the plots do NOT depend on the
# GPU-heavy D scoring having finished for a given cell/fraction.
REPO = "/home/mila/a/ahmedm/gradmatch_selection"
SCRATCH = "/network/scratch/a/ahmedm"
DATASET = "cifar10"
ARCHITECTURES = ["resnet18", "vgg16", "inception"]
FRACTIONS = [0.1, 0.2, 0.4, 0.5]        # which fraction-tagged CSVs to merge
KL_K = 1                                # checkpoint count of "ours" (the paper's run)
KL_NAME = f"Gradient Swap, K={KL_K} (ours)"
OUT_DIR = os.path.join(SCRATCH, "data_selection_baselines", "plots")


In [6]:
import os
import sys

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

tol_bright = ['#4477AA', '#EE6677', '#228833', '#AA3377', '#66CCEE', '#CCBB44', '#000000']
linestyles = ['-', '--', '-.', ':', '-', '--', '-.']
markers = ["*", "o", "s", "^", "v", "D", "<", ">"]

# ---------------------------------------------------------------- journal style
plt.rcParams.update({
    # fonts: serif (Times-like), embedded as TrueType so journals accept the PDF
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "STIXGeneral", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,

    # font sizes (bigger)
    "font.size": 22,
    "axes.labelsize": 28,
    "xtick.labelsize": 24,
    "ytick.labelsize": 24,
    "legend.fontsize": 20,

    # axes / ticks
    "axes.linewidth": 1.8,
    "axes.labelpad": 8,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 7,
    "xtick.major.width": 1.8,
    "xtick.major.pad": 6,
    "legend.frameon": True,
    "legend.fancybox": False,
    "legend.edgecolor": "0.2",
    "legend.framealpha": 0.95,
    "legend.handlelength": 2.6,
    "legend.borderpad": 0.5,
    "legend.labelspacing": 0.35,

    # output
    "savefig.dpi": 300,
})

sys.path.insert(0, os.path.join(REPO, "src"))
from data_selection_swap.analysis.comparison import discover_methods
from data_selection_swap.analysis.score_subsets import collect_rows

SEEDS = [42, 43, 44, 45, 46]
os.makedirs(OUT_DIR, exist_ok=True)

for architecture in ARCHITECTURES:
    # ------------------------------------------------------------- load accuracies
    # One row per saved run straight from *_metrics.json (fraction, method, K,
    # seed, acc). _from_* warm starts are their own paper table, not part of
    # the cold-start accuracy curves, so they are excluded here.
    methods = [m for m in sorted(discover_methods(SCRATCH, DATASET, architecture))
               if "_from_" not in m]
    rows = collect_rows(SCRATCH, DATASET, architecture, FRACTIONS,
                        SEEDS, methods, checkpoints=None)
    df = pd.DataFrame(rows).rename(columns={"k": "num_checkpoints_tag"})
    df = df.dropna(subset=["acc"]).copy()
    cov = df.groupby(["method", "fraction"])["seed"].count()
    print(f"{architecture}: {len(df)} accuracy points "
          f"(seeds per point: {int(cov.min())}-{int(cov.max())})")

    # keep the KL run at KL_K checkpoints, keep all baselines (no K tag)
    is_kl = df["method"].str.startswith("kl_faithful")
    df = df[~is_kl | (df["num_checkpoints_tag"] == KL_K)].copy()
    if (df["method"].str.startswith("kl_faithful")).sum() == 0:
        print(f"  WARNING: no kl_faithful rows with num_checkpoints_tag == {KL_K}")
    df.loc[df["method"].str.startswith("kl_faithful"), "method"] = KL_NAME
    df.loc[df["method"].str.startswith("facility_location"), "method"] = "Facility Location"
    df.loc[df["method"].str.startswith("graphcut"), "method"] = "GraphCut"
    df.loc[df["method"].str.startswith("herding"), "method"] = "Herding"
    df.loc[df["method"].str.startswith("moderate"), "method"] = "Moderate Coreset"
    df.loc[df["method"].str.startswith("moso"), "method"] = "MoSo"
    df.loc[df["method"].str.startswith("random"), "method"] = "Random"
    df.loc[df["method"].str.startswith("el2n"), "method"] = "EL2N"
    df["frac"] = df["fraction"]          # the scorer records the pool-relative fraction
    # 'acc' is the final-epoch test accuracy in % (the old notebook's
    # budget/TRAIN_SIZE fraction and 100-error columns no longer exist)

    # drop EL2N's first (smallest-fraction) point
    el2n_first = df.loc[df["method"] == "EL2N", "frac"].min()
    if pd.notna(el2n_first):
        df = df[~((df["method"] == "EL2N") & (df["frac"] == el2n_first))]

    # ------------------------------------------------------------- aggregate
    agg = (df.groupby(["method", "frac"])["acc"]
             .agg(mean="mean", std="std", n="count")
             .reset_index())
    agg["std"] = agg["std"].fillna(0.0)
    print(agg.to_string(index=False))        # n = number of seeds per point

    # ------------------------------------------------------------- style
    # ours gets slot 0; baselines follow, with EL2N last (it is mostly off-plot)
    others = [m for m in sorted(agg["method"].unique()) if m not in (KL_NAME, "EL2N")]
    if "EL2N" in set(agg["method"]):
        others.append("EL2N")
    order = [KL_NAME] + others

    def get_style(idx):
        # 8 curves but 7 colors: on wrap-around, shift the linestyle so that
        # (color, linestyle) pairs stay unique
        return dict(color=tol_bright[idx % 7],
                    ls=linestyles[(idx + idx // 7) % 7],
                    marker=markers[idx % len(markers)])

    fracs = sorted(agg["frac"].unique())

    # ------------------------------------------------------------- plot
    fig, ax = plt.subplots(figsize=(10, 7.5))

    for idx, m in enumerate(order):
        d = agg[agg["method"] == m].sort_values("frac")
        s = get_style(idx)
        is_ours = (m == KL_NAME)
        ax.plot(d["frac"], d["mean"], color=s["color"], linestyle=s["ls"],
                marker=s["marker"], lw=3.4 if is_ours else 2.6,
                ms=18 if is_ours else 10, label=m,
                markeredgecolor=s["color"], zorder=10 if is_ours else 3)
        ax.fill_between(d["frac"], d["mean"] - d["std"], d["mean"] + d["std"],
                        color=s["color"], alpha=0.15, lw=0, zorder=2)

    # y-limits set from everything except EL2N, so EL2N runs off the bottom
    ref = agg[agg["method"] != "EL2N"]
    lo = (ref["mean"] - ref["std"]).min()
    hi = (ref["mean"] + ref["std"]).max()
    pad = 0.06 * (hi - lo)
    ax.set_ylim(lo - pad, hi + pad)

    ax.set_xticks(fracs)
    ax.set_xlim(min(fracs) - 0.04, max(fracs) + 0.04)
    ax.set_xlabel("Fraction of the training set")
    ax.set_ylabel("Test accuracy (%)")
    ax.grid(True, linestyle="--", linewidth=0.8, alpha=0.4)
    ax.set_axisbelow(True)
    ax.legend(loc="lower right")

    fig.tight_layout()
    out_pdf = os.path.join(OUT_DIR, f"{DATASET}_{architecture}_accuracy.pdf")
    fig.savefig(out_pdf, bbox_inches="tight", pad_inches=0.05)
    plt.close(fig)
    print(f"Saved {out_pdf}")


resnet18: d_exact_cifar10_resnet18_f0.1.csv
                   method  frac   mean      std  n
        Facility Location   0.1 71.324 1.004405  5
Gradient Swap, K=1 (ours)   0.1 74.474 0.638498  5
                 GraphCut   0.1 74.012 0.312602  5
                  Herding   0.1 74.862 0.883895  5
                     MoSo   0.1 72.820 0.901554  5
         Moderate Coreset   0.1 76.522 0.252527  5
                   Random   0.1 75.266 0.791726  5
Saved /network/scratch/a/ahmedm/data_selection_baselines/plots/cifar10_resnet18_accuracy.pdf
vgg16: d_exact_cifar10_vgg16_f0.1.csv, d_exact_cifar10_vgg16_f0.2.csv, d_exact_cifar10_vgg16_f0.4.csv
                   method  frac   mean      std  n
                     EL2N   0.2 61.302 2.918111  5
                     EL2N   0.4 90.878 0.207292  5
        Facility Location   0.1 75.234 0.895059  5
        Facility Location   0.2 82.884 0.268198  5
        Facility Location   0.4 89.316 0.307376  5
Gradient Swap, K=1 (ours)   0.1 77.064 0.156461